# Notebook 04: Counseling Session Effectiveness (Explanatory)

## Section 1 — Problem Framing

**Business Problem:** Identify which counseling session types and interventions are most effective at improving residents' emotional state, so counselors can optimize their approach.

**Approach:** Explanatory OLS regression — we interpret coefficients to understand *which* interventions and session configurations drive the largest emotional improvements.

**Target Variable:** `emotional_improvement` = end_score - start_score (ordinal encoding of emotional states)

**Ordinal Encoding (per course spec):**
- Distressed=1, Withdrawn=2, Angry=3, Anxious=4, Sad=5, Calm=6, Hopeful=7, Happy=8

A positive `emotional_improvement` means the resident moved to a more positive emotional state during the session.

**Stakeholders:** Counselors, case managers

## Section 2 — Data Acquisition and Preparation

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy.stats import shapiro
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")

All imports successful


In [2]:
# Load data
pr = pd.read_csv('../../data/lighthouse_csv_v7/process_recordings.csv')
print(f"Dataset shape: {pr.shape}")
print(f"\nColumns: {list(pr.columns)}")
print(f"\nEmotional states observed: {pr['emotional_state_observed'].value_counts().to_dict()}")
print(f"\nEmotional states end: {pr['emotional_state_end'].value_counts().to_dict()}")
print(f"\nSession types: {pr['session_type'].value_counts().to_dict()}")

Dataset shape: (2819, 15)

Columns: ['recording_id', 'resident_id', 'session_date', 'social_worker', 'session_type', 'session_duration_minutes', 'emotional_state_observed', 'emotional_state_end', 'session_narrative', 'interventions_applied', 'follow_up_actions', 'progress_noted', 'concerns_flagged', 'referral_made', 'notes_restricted']

Emotional states observed: {'Sad': 499, 'Calm': 476, 'Anxious': 462, 'Angry': 392, 'Hopeful': 391, 'Withdrawn': 356, 'Happy': 150, 'Distressed': 93}

Emotional states end: {'Hopeful': 1178, 'Calm': 896, 'Happy': 435, 'Sad': 154, 'Anxious': 137, 'Withdrawn': 19}

Session types: {'Individual': 1805, 'Group': 1014}


In [3]:
# Ordinal encode emotional states (per course spec)
emotion_map = {
    'Distressed': 1, 'Withdrawn': 2, 'Angry': 3, 'Anxious': 4,
    'Sad': 5, 'Calm': 6, 'Hopeful': 7, 'Happy': 8
}

pr['start_score'] = pr['emotional_state_observed'].map(emotion_map)
pr['end_score'] = pr['emotional_state_end'].map(emotion_map)
pr['emotional_improvement'] = pr['end_score'] - pr['start_score']

print("Emotional state encoding applied:")
print(f"  Mapping: {emotion_map}")
print(f"\n  start_score stats: mean={pr['start_score'].mean():.2f}, std={pr['start_score'].std():.2f}")
print(f"  end_score stats: mean={pr['end_score'].mean():.2f}, std={pr['end_score'].std():.2f}")
print(f"  emotional_improvement stats: mean={pr['emotional_improvement'].mean():.2f}, std={pr['emotional_improvement'].std():.2f}")
print(f"\n  Improvement range: [{pr['emotional_improvement'].min()}, {pr['emotional_improvement'].max()}]")
print(f"  % positive improvement: {(pr['emotional_improvement'] > 0).mean()*100:.1f}%")
print(f"  % no change: {(pr['emotional_improvement'] == 0).mean()*100:.1f}%")
print(f"  % negative: {(pr['emotional_improvement'] < 0).mean()*100:.1f}%")

Emotional state encoding applied:
  Mapping: {'Distressed': 1, 'Withdrawn': 2, 'Angry': 3, 'Anxious': 4, 'Sad': 5, 'Calm': 6, 'Hopeful': 7, 'Happy': 8}

  start_score stats: mean=4.65, std=1.86
  end_score stats: mean=6.55, std=1.05
  emotional_improvement stats: mean=1.89, std=1.45

  Improvement range: [-1, 5]
  % positive improvement: 82.2%
  % no change: 15.0%
  % negative: 2.7%


In [4]:
# Split comma-separated interventions_applied into binary indicators
for intervention in ['Caring', 'Teaching', 'Legal Services', 'Healing']:
    safe_name = intervention.replace(' ', '_')
    pr[f'has_{safe_name}'] = pr['interventions_applied'].str.contains(intervention, na=False).astype(int)

print("Intervention binary indicators created:")
for intervention in ['Caring', 'Teaching', 'Legal_Services', 'Healing']:
    col = f'has_{intervention}'
    print(f"  {col}: {pr[col].sum()} sessions ({pr[col].mean()*100:.1f}%)")

Intervention binary indicators created:
  has_Caring: 1400 sessions (49.7%)
  has_Teaching: 1448 sessions (51.4%)
  has_Legal_Services: 1380 sessions (49.0%)
  has_Healing: 1427 sessions (50.6%)


In [5]:
# Prepare features for OLS
# Features: session_type, session_duration_minutes, intervention indicators, start_score (control)
categorical_features = ['session_type']
numeric_features = ['session_duration_minutes', 'start_score',
                    'has_Caring', 'has_Teaching', 'has_Legal_Services', 'has_Healing']

# One-hot encode session_type
features_df = pd.get_dummies(pr[categorical_features + numeric_features],
                              columns=categorical_features, drop_first=True, dtype=int)

y = pr['emotional_improvement']

# Drop any rows with NaN
mask = features_df.notna().all(axis=1) & y.notna()
features_df = features_df[mask]
y = y[mask]

X = sm.add_constant(features_df)

print(f"Feature matrix shape: {X.shape}")
print(f"Target (emotional_improvement) shape: {y.shape}")
print(f"\nFeatures used:")
for col in X.columns[1:]:
    print(f"  - {col}")

Feature matrix shape: (2819, 8)
Target (emotional_improvement) shape: (2819,)

Features used:
  - session_duration_minutes
  - start_score
  - has_Caring
  - has_Teaching
  - has_Legal_Services
  - has_Healing
  - session_type_Individual


In [6]:
# Train/test split for sklearn deployment
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    pr[mask][categorical_features + numeric_features],
    y, test_size=0.2, random_state=42
)
print(f"Train size: {len(X_train_raw)}, Test size: {len(X_test_raw)}")

Train size: 2255, Test size: 564


## Section 3 — Exploration

In [7]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution of emotional_improvement
axes[0, 0].hist(y, bins=15, edgecolor='black', alpha=0.7, color='teal')
axes[0, 0].set_title('Distribution of Emotional Improvement')
axes[0, 0].set_xlabel('Emotional Improvement (end - start)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(x=0, color='red', linestyle='--', label='No change')
axes[0, 0].legend()

# Mean improvement by session type
session_means = pr[mask].groupby('session_type')['emotional_improvement'].mean()
axes[0, 1].bar(session_means.index, session_means.values, color='teal', edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Mean Emotional Improvement by Session Type')
axes[0, 1].set_ylabel('Mean Improvement')

# Mean improvement by intervention
intervention_cols = ['has_Caring', 'has_Teaching', 'has_Legal_Services', 'has_Healing']
intervention_means = []
for col in intervention_cols:
    with_intervention = pr[mask][pr[mask][col] == 1]['emotional_improvement'].mean()
    without = pr[mask][pr[mask][col] == 0]['emotional_improvement'].mean()
    intervention_means.append({'Intervention': col.replace('has_', ''),
                                'With': with_intervention, 'Without': without})
int_df = pd.DataFrame(intervention_means)
x_pos = np.arange(len(int_df))
width = 0.35
axes[1, 0].bar(x_pos - width/2, int_df['With'], width, label='With intervention', color='teal', alpha=0.7)
axes[1, 0].bar(x_pos + width/2, int_df['Without'], width, label='Without', color='salmon', alpha=0.7)
axes[1, 0].set_xticks(x_pos)
axes[1, 0].set_xticklabels(int_df['Intervention'], rotation=45)
axes[1, 0].set_title('Mean Improvement by Intervention')
axes[1, 0].set_ylabel('Mean Improvement')
axes[1, 0].legend()

# Session duration vs improvement
axes[1, 1].scatter(pr[mask]['session_duration_minutes'], y, alpha=0.2, color='teal')
axes[1, 1].set_title('Session Duration vs Emotional Improvement')
axes[1, 1].set_xlabel('Session Duration (minutes)')
axes[1, 1].set_ylabel('Emotional Improvement')

plt.tight_layout()
plt.savefig('../../ml-pipelines/notebooks/04_exploration.png', dpi=100, bbox_inches='tight')
plt.show()
print("Exploration plots generated")

Exploration plots generated


## Section 4 — Modeling (Explanatory OLS)

In [8]:
# Fit OLS model (explanatory — for coefficient interpretation)
model = sm.OLS(y, X).fit()
print(model.summary())

                              OLS Regression Results                             
Dep. Variable:     emotional_improvement   R-squared:                       0.689
Model:                               OLS   Adj. R-squared:                  0.688
Method:                    Least Squares   F-statistic:                     888.9
Date:                   Mon, 06 Apr 2026   Prob (F-statistic):               0.00
Time:                           15:15:47   Log-Likelihood:                -3408.3
No. Observations:                   2819   AIC:                             6833.
Df Residuals:                       2811   BIC:                             6880.
Df Model:                              7                                         
Covariance Type:               nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------

In [9]:
# VIF check for multicollinearity
vif_data = pd.DataFrame({
    'Feature': X.columns[1:],  # Skip constant
    'VIF': [variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])]
})
vif_data = vif_data.sort_values('VIF', ascending=False)
print("Variance Inflation Factors:")
print(vif_data.to_string(index=False))

high_vif = vif_data[vif_data['VIF'] > 10]
if len(high_vif) > 0:
    print(f"\nWARNING: {len(high_vif)} features with VIF > 10 detected.")
    print("Dropping high-VIF features and re-fitting...")
    drop_cols = high_vif['Feature'].tolist()
    X_clean = X.drop(columns=drop_cols)
    model = sm.OLS(y, X_clean).fit()
    print("\nRe-fitted model summary:")
    print(model.summary())
    vif_data2 = pd.DataFrame({
        'Feature': X_clean.columns[1:],
        'VIF': [variance_inflation_factor(X_clean.values, i) for i in range(1, X_clean.shape[1])]
    })
    print("\nUpdated VIF values:")
    print(vif_data2.sort_values('VIF', ascending=False).to_string(index=False))
else:
    print("\nAll VIF values <= 10. No multicollinearity issues detected.")

Variance Inflation Factors:
                 Feature      VIF
 session_type_Individual 1.367030
session_duration_minutes 1.364984
            has_Teaching 1.055861
      has_Legal_Services 1.053631
             has_Healing 1.048846
              has_Caring 1.045752
             start_score 1.002413

All VIF values <= 10. No multicollinearity issues detected.


### Coefficient Interpretation

The OLS model reveals which session factors significantly predict emotional improvement:
- **session_type (Group vs Individual):** Does the setting affect improvement?
- **session_duration_minutes:** Do longer sessions produce better outcomes?
- **start_score:** Controlling for baseline emotional state (higher baselines may have less room to improve)
- **Interventions (Caring, Teaching, Legal Services, Healing):** Which therapeutic approaches drive the most improvement?

## Section 5 — Evaluation

In [10]:
# Model fit statistics
print(f"R-squared: {model.rsquared:.4f}")
print(f"Adjusted R-squared: {model.rsquared_adj:.4f}")
y_pred_ols = model.fittedvalues
rmse = np.sqrt(mean_squared_error(y, y_pred_ols))
print(f"RMSE: {rmse:.4f}")
print(f"F-statistic: {model.fvalue:.2f}, p-value: {model.f_pvalue:.2e}")

R-squared: 0.6888
Adjusted R-squared: 0.6880
RMSE: 0.8107
F-statistic: 888.91, p-value: 0.00e+00


In [11]:
# Residual diagnostics
residuals = model.resid
fitted = model.fittedvalues

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Residuals vs Fitted
axes[0].scatter(fitted, residuals, alpha=0.2, color='teal')
axes[0].axhline(y=0, color='black', linestyle='--')
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted Values')

# Q-Q plot
sm.qqplot(residuals, line='45', ax=axes[1])
axes[1].set_title('Q-Q Plot of Residuals')

# Histogram
axes[2].hist(residuals, bins=30, edgecolor='black', alpha=0.7, color='teal')
axes[2].set_title('Distribution of Residuals')
axes[2].set_xlabel('Residual')

plt.tight_layout()
plt.savefig('../../ml-pipelines/notebooks/04_residuals.png', dpi=100, bbox_inches='tight')
plt.show()

In [12]:
# Shapiro-Wilk test for normality
if len(residuals) > 5000:
    stat, p_val = shapiro(residuals.sample(5000, random_state=42))
else:
    stat, p_val = shapiro(residuals)
print(f"Shapiro-Wilk Test: statistic={stat:.4f}, p-value={p_val:.4f}")
if p_val < 0.05:
    print("Residuals are NOT normally distributed (p < 0.05).")
    print("Note: With ordinal target data, non-normality is expected. OLS coefficients remain interpretable.")
else:
    print("Residuals appear normally distributed (p >= 0.05).")

# Breusch-Pagan test for homoscedasticity
X_model = model.model.exog
bp_test = het_breuschpagan(residuals, X_model)
labels = ['LM Statistic', 'LM p-value', 'F-Statistic', 'F p-value']
print(f"\nBreusch-Pagan Test:")
for label, val in zip(labels, bp_test):
    print(f"  {label}: {val:.4f}")
if bp_test[1] < 0.05:
    print("Heteroscedasticity detected (p < 0.05). Consider robust standard errors.")
else:
    print("No evidence of heteroscedasticity (p >= 0.05).")

Shapiro-Wilk Test: statistic=0.9570, p-value=0.0000
Residuals are NOT normally distributed (p < 0.05).
Note: With ordinal target data, non-normality is expected. OLS coefficients remain interpretable.



Breusch-Pagan Test:
  LM Statistic: 275.8267
  LM p-value: 0.0000
  F-Statistic: 43.5535
  F p-value: 0.0000
Heteroscedasticity detected (p < 0.05). Consider robust standard errors.


In [13]:
# Coefficient table with confidence intervals
coef_table = pd.DataFrame({
    'Coefficient': model.params,
    'Std Error': model.bse,
    'p-value': model.pvalues,
    'CI Lower': model.conf_int()[0],
    'CI Upper': model.conf_int()[1]
})
coef_table = coef_table.drop('const', errors='ignore')
coef_table = coef_table.sort_values('p-value')
print("Coefficient Table (sorted by significance):")
print(coef_table.to_string())

print(f"\nSignificant features (p < 0.05): {(coef_table['p-value'] < 0.05).sum()} of {len(coef_table)}")

Coefficient Table (sorted by significance):
                          Coefficient  Std Error   p-value  CI Lower  CI Upper
start_score                 -0.647329   0.008222  0.000000 -0.663451 -0.631207
has_Teaching                 0.062526   0.031435  0.046793  0.000887  0.124165
session_duration_minutes    -0.001417   0.000799  0.076260 -0.002984  0.000150
has_Healing                 -0.048442   0.031321  0.122070 -0.109857  0.012973
has_Caring                  -0.031162   0.031273  0.319131 -0.092483  0.030160
has_Legal_Services          -0.018990   0.031397  0.545336 -0.080554  0.042574
session_type_Individual     -0.003794   0.037252  0.918883 -0.076838  0.069250

Significant features (p < 0.05): 2 of 7


## Section 6 — Causal Analysis

In [14]:
# Significant coefficients interpretation
sig = coef_table[coef_table['p-value'] < 0.05].copy()

print("SIGNIFICANT FACTORS affecting emotional improvement (p < 0.05):")
print("=" * 80)
for idx, row in sig.iterrows():
    direction = "+" if row['Coefficient'] > 0 else ""
    label = idx.replace('session_type_', 'Session: ').replace('has_', 'Intervention: ')
    print(f"  {label}: {direction}{row['Coefficient']:.4f} (p={row['p-value']:.4f})")
    if row['Coefficient'] > 0:
        print(f"    -> Associated with {abs(row['Coefficient']):.2f} MORE emotional improvement")
    else:
        print(f"    -> Associated with {abs(row['Coefficient']):.2f} LESS emotional improvement")
print()
print("Interpretation: Each coefficient represents the change in emotional improvement")
print("associated with a one-unit increase in the factor, holding other factors constant.")

SIGNIFICANT FACTORS affecting emotional improvement (p < 0.05):
  start_score: -0.6473 (p=0.0000)
    -> Associated with 0.65 LESS emotional improvement
  Intervention: Teaching: +0.0625 (p=0.0468)
    -> Associated with 0.06 MORE emotional improvement

Interpretation: Each coefficient represents the change in emotional improvement
associated with a one-unit increase in the factor, holding other factors constant.


### Confounders and Limitations

1. **Self-selection into session types:** Residents or counselors may choose session types based on the resident's current state, creating selection bias. More distressed residents may be assigned to individual sessions.
2. **Counselor skill differences:** The model does not account for individual counselor skill or experience. Some counselors may be more effective regardless of intervention type.
3. **Resident baseline differences:** While we control for `start_score`, residents differ in resilience, trauma history, and time in the program.
4. **Intervention combinations:** The binary indicators don't capture interaction effects between interventions (e.g., Caring + Healing together may be synergistic).
5. **Ordinal scale limitations:** Treating ordinal emotional states as interval data assumes equal spacing between levels, which may not hold.

### Recommendations

Based on the significant coefficients:
- **Prioritize intervention types with the largest positive coefficients** in treatment plans
- **Consider session type effects** when scheduling — if group sessions show higher improvement, increase group availability
- **Monitor session duration** — if longer sessions show diminishing returns, optimize scheduling
- **Control for baseline state** when evaluating counselor performance

## Section 7 - Deployment

**Deployment Architecture:** Pre-computed predictions written to PostgreSQL.

This model is deployed as an offline batch pipeline. The production workflow is:

1. **ETL:** `jobs/etl_counseling.py` reads process_recordings from PostgreSQL, engineers features (emotional state scoring, intervention counts), and writes the `ml_counseling_features` table.
2. **Train:** `jobs/train_counseling.py` trains an OLS LinearRegression pipeline, saves the model as `artifacts/counseling.sav` along with `metadata.json` and `metrics.json`.
3. **Inference:** `jobs/run_inference_counseling.py` loads the trained model, predicts emotional improvement for all sessions, and writes results to the `counseling_predictions` table in PostgreSQL.

The .NET backend queries `counseling_predictions` via EF Core. The frontend fetches insights from `GET /api/predictions/counseling`.

**Model type:** Explanatory (OLS regression)

**Approach:** Explanatory -- which session factors drive emotional improvement. The sklearn Pipeline wraps the OLS model for inference consistency.

In [15]:
# For Flask API deployment: Train sklearn LinearRegression Pipeline
cat_features_deploy = ['session_type']
num_features_deploy = ['session_duration_minutes', 'start_score',
                        'has_Caring', 'has_Teaching', 'has_Legal_Services', 'has_Healing']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_features_deploy),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_features_deploy)
])

sklearn_pipeline = SkPipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

sklearn_pipeline.fit(X_train_raw, y_train)

# Evaluate on test set
y_pred_sk = sklearn_pipeline.predict(X_test_raw)
print(f"sklearn Pipeline Test R-squared: {r2_score(y_test, y_pred_sk):.4f}")
print(f"sklearn Pipeline Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_sk)):.4f}")

sklearn Pipeline Test R-squared: 0.6750
sklearn Pipeline Test RMSE: 0.8247


In [ ]:
# Save sklearn pipeline as .sav to artifacts/
joblib.dump(sklearn_pipeline, '../artifacts/counseling.sav')
print("Model saved to: ../artifacts/counseling.sav")

print("\nProduction scripts:")
print("  ETL:       jobs/etl_counseling.py")
print("  Train:     jobs/train_counseling.py")
print("  Inference: jobs/run_inference_counseling.py")
print("\nPredictions are pre-computed to PostgreSQL table: counseling_predictions")

In [17]:
# Expected input features for prediction
print("Expected input features (as DataFrame columns):")
print(f"  Categorical: {cat_features_deploy}")
print(f"  Numeric: {num_features_deploy}")
print()

# Example prediction
example = pd.DataFrame([{
    'session_type': 'Individual',
    'session_duration_minutes': 60,
    'start_score': 3,  # Angry
    'has_Caring': 1,
    'has_Teaching': 0,
    'has_Legal_Services': 0,
    'has_Healing': 1
}])
pred = loaded.predict(example)
print(f"Example prediction (emotional_improvement): {pred[0]:.2f}")
print(f"If start_score=3 (Angry), predicted end_score would be: {3 + pred[0]:.2f}")

Expected input features (as DataFrame columns):
  Categorical: ['session_type']
  Numeric: ['session_duration_minutes', 'start_score', 'has_Caring', 'has_Teaching', 'has_Legal_Services', 'has_Healing']

Example prediction (emotional_improvement): 2.91
If start_score=3 (Angry), predicted end_score would be: 5.91
